# L3 — Max-2-SAT: rules that can be broken, at a price

**2026 Quantum Accelerator · Day 1 · QUBO ladder · step 3 of 5 · ~15 minutes**

**New on this step:** the coefficients come from *logic*. A clause fails only when both of its
literals are false, and that product is already quadratic. Signs are mixed, Q is as sparse as
the clause list, and the **constant term must be kept**, because the optimum value itself carries
information (0 means every clause can be satisfied).

**The problem.** Variables are true or false; a clause is "this OR that"; not all clauses can be
true at once; make as much clause weight true as possible. **Why it matters.** Every "if-then"
business rule becomes a clause, and this is the bridge from pure objectives to constraints.
**Plan.** The slide instance by hand, then 100 variables and 400 clauses.

---
## 1. The slide instance

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import helpers_ladder as h

labels = ["x1", "x2", "x3"]
clauses = h.max2sat_slide()          # (i, sign_i, j, sign_j, weight); sign +1 = x, -1 = NOT x
clauses[0] = (0, 1, 1, 1, 1)         # <-- CHANGE ME: weight of C1, 1 -> 4 (which clause is sacrificed now?)
print(f"{'clause':<24}{'weight':>7}   fails when")
for k, c in enumerate(clauses, 1):
    i, si, j, sj, w = c
    when = f"{labels[i]} = {0 if si == 1 else 1}, {labels[j]} = {0 if sj == 1 else 1}"
    print(f"C{k}: {h.clause_text(c, labels).split('  (')[0]:<20}{w:>7}   {when}")

clause                   weight   fails when
C1: x1 OR x2                  1   x1 = 0, x2 = 0
C2: x1 OR NOT x2              2   x1 = 0, x2 = 1
C3: NOT x1 OR NOT x2          3   x1 = 1, x2 = 1
C4: NOT x1 OR NOT x3          2   x1 = 1, x3 = 1
C5: x2 OR x3                  2   x2 = 0, x3 = 0


---
## 2. Q by hand

Each clause costs its weight `w` when it is violated, i.e. when both literals are false:
`w · (1 − l₁)(1 − l₂)`. For a positive literal `(1 − l)` is `1 − x`; for a negated one it is `x`.
Expanding the product gives linear terms (diagonal), one cross term (off-diagonal) and a constant.

In [2]:
Q = np.zeros((3, 3)); CONST = 0
for (i, si, j, sj, w) in clauses:
    # (1 - literal) is (1 - x) for a positive literal and x for a negated one
    A = {"c": 1, i: -1} if si == 1 else {i: 1}
    B = {"c": 1, j: -1} if sj == 1 else {j: 1}
    for ka, va in A.items():
        for kb, vb in B.items():
            if ka == "c" and kb == "c": CONST += w * va * vb
            elif ka == "c":             Q[kb, kb] += w * va * vb
            elif kb == "c":             Q[ka, ka] += w * va * vb
            else:                       Q[ka, kb] += w * va * vb / 2; Q[kb, ka] += w * va * vb / 2
h.print_matrix(Q, labels, "Q")
print("CONST =", CONST)

Q
      x1  x2  x3
  x1  -1   1   1
  x2   1  -1   1
  x3   1   1  -2
CONST = 3


**What you see.** Walk clause C2 (`x1 OR NOT x2`, weight 2) through the loop: it is violated
when `x1 = 0` and `x2 = 1`, so the cost is `2·(1 − x1)·x2 = 2x2 − 2x1x2`. Now look at the
(1,2) pair: **+1** from C1, **−2** from C2, **+3** from C3, net **+2**, so `Q12 = 1`
(half on each side of the diagonal). Mixed signs, straight from the logic.

---
## 3. Every answer

In [3]:
table = h.exact_table(Q, labels, const=CONST)
table["violated weight"] = table["objective"]
table["violated"] = [
    ",".join(f"C{k}" for k, c in enumerate(clauses, 1) if h.violated_weight([c], row.values[:3]) > 0) or "-"
    for _, row in table.iterrows()]
table

,x1,x2,x3,energy,objective,violated weight,violated
0,0,0,1,-2,1,1,C1
1,0,1,0,-1,2,2,C2
2,0,1,1,-1,2,2,C2
3,1,0,0,-1,2,2,C5
4,1,0,1,-1,2,2,C4
5,0,0,0,0,3,3,"C1,C5"
6,1,1,0,0,3,3,C3
7,1,1,1,2,5,5,"C3,C4"


**What you see.** The optimum is unique: `x = (0, 0, 1)` violates only C1 (weight 1). The best
value is 1, not 0, so the instance is **unsatisfiable**, and the solver sacrifices the cheapest
clause. This is why the constant matters here and did not in L2: `energy + CONST` *is* the
violated weight, and "0 or not" answers a yes/no question about the whole rule set.

---
## 4. The same Q from PyQUBO

In [4]:
from pyqubo import Binary

x = [Binary(f"x{i+1}") for i in range(3)]
lit = lambda i, s: x[i] if s == 1 else 1 - x[i]
unsat = sum(w * (1 - lit(i, si)) * (1 - lit(j, sj)) for (i, si, j, sj, w) in clauses)
model = unsat.compile()
qubo, offset = model.to_qubo()
print(np.allclose(Q, h.symmetric_from_qubo_dict(qubo, labels)), offset)   # True 3.0

True 3.0


---
## 5. Now too big for a slide: 100 variables, 400 clauses

In [5]:
clauses_big = h.max2sat_instance(n=100, m=400, seed=1)       # <-- CHANGE ME (stretch): m=400 -> 600
W_tot = sum(c[4] for c in clauses_big)
Q_big, CONST_big = h.max2sat_Q(clauses_big, 100)
ss, dt = h.anneal(Q_big)

R = h.read_matrix(ss, 100)
viol = np.array([h.violated_weight(clauses_big, x) for x in R])
rng = np.random.default_rng(0)
rand = np.array([h.violated_weight(clauses_big, x) for x in rng.integers(0, 2, (100, 100))])
print(f"total clause weight W = {W_tot}, constant = {CONST_big:.0f}")
print(f"SA: violated weight best {viol.min()}, mean {viol.mean():.0f}, worst {viol.max()}; "
      f"{(viol == viol.min()).sum()} of 100 reads at the best   ({dt:.2f} s)")
print(f"random assignments: mean violated weight {rand.mean():.0f}")

total clause weight W = 1219, constant = 309
SA: violated weight best 68, mean 69, worst 73; 54 of 100 reads at the best   (0.26 s)
random assignments: mean violated weight 304


In [6]:
ref = h.cpsat_max2sat(clauses_big, 100, time_limit=30)       # <-- CHANGE ME
print(f"CP-SAT: violated {ref['violated']:.0f}, bound {ref['bound_violated']:.0f}, "
      f"status {ref['status']}, {ref['seconds']:.2f} s")

plt.hist(viol, bins=range(viol.min(), viol.max() + 2), color="#1f77b4", alpha=0.8)
plt.axvline(ref["violated"], color="crimson", ls="--", label=f"CP-SAT, 30 s: {ref['violated']:.0f}")
plt.axvline(rand.mean(), color="grey", ls=":", label=f"random mean: {rand.mean():.0f}")
plt.xlabel("violated weight of one SA read"); plt.ylabel("reads"); plt.legend(); plt.show()

CP-SAT: violated 68, bound 0, status FEASIBLE, 30.00 s


**What you see.** SA and the exact solver agree on 68, but neither can prove it: the exact
solver's bound stayed at 0. Weighted Max-SAT at this clause density is a known hard regime for
proof-based methods, and specialised Max-SAT solvers (not CP-SAT) are the state of the art there.
This notebook does **not** show annealing beating anything, only reaching the same answer
cheaply. (A random assignment violates about a quarter of the weight, 1 219 / 4 ≈ 305.)

---
## 6. What changed on this step

- The coefficients were derived from logic: one violated-clause product per rule.
- Signs are mixed and Q is sparse (one pair per clause).
- The **constant** (3 on the slide, 309 at scale) is kept: `energy + CONST` is the violated weight.
- At n = 100 SA reaches the same value as a 30-second exact run, but nobody can prove it optimal.
- Next, **L4**: the rules become *hard* ("never two neighbours"), with a reward on top and a penalty to tune.

---
## Stretch

**A.** Remove C3. Is the instance now satisfiable (objective 0)?

**B.** Set every weight to 1 on the big instance. How many reads tie for best, and why are ties
more common without weights?

**C.** Max-Cut as Max-2-SAT: take the L1 triangle with weights made positive (8, 2, 3), write each
edge as two clauses `(x_i OR x_j)` and `(NOT x_i OR NOT x_j)` with the edge weight, and check that
`max2sat_Q` reproduces L1's Q up to a constant. Negative edge weights have no clause equivalent — why not?